# Generalized E3SMLE and CESM-SMYLE Atmospheric Skill Maps

This notebook is adapted from `YeagerEA_GMD2022_revision1/f01_fB01_fB03.ipynb` and `YeagerEA_GMD2022_revision1/f02_fB02_fB04.ipynb`. It preserves their lead-time seasonal ACC methodology and map-analysis structure while refactoring data access for the E3SM and CESM-SMYLE comparison and generalizing the workflow across fields.

This notebook computes and plots seasonal skill for `TREFHT`, `TS`, `PRECT`, `PSL`, or ocean-only `SST` using the same workflow.

Change only one line in the configuration cell below:

```python
# field = "TREFHT"  # surface air temperature
# field = "TS"      # all-surface radiative/skin temperature
# field = "PRECT" # precipitation
# field = "PSL"   # sea-level pressure
# field = "SST"   # sea-surface temperature (HadISST2 ocean mask)
```

Variable-specific observation product, observation variable name, unit conversion, labels, and cache filenames are handled through `VAR_CONFIG`.


In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import xarray as xr 
import numpy as np  
import cftime
import copy
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import xesmf as xe
import xskillscore as xs
%matplotlib inline

# import ESP-Lab modules
#from esp_lab import data_access_smyle.py as data_access
from esp_lab import data_access_e3sm as data_access
from esp_lab import data_access_obs as obs_access
from esp_lab import stats

# import plotting and other utilities from esp_lab.utils (formerly SMYLEutils)
from esp_lab.utils import calendar_utils as cal
from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import colorbar_utils as cbars
from esp_lab.utils import regrid_utils as regrid
from esp_lab.utils import mov_utils as mov
from esp_lab.paths import (
    S2D_DIAG_ROOT,
    E3SMLE_DIAG_DIR,
    CESM_SMYLE_DIAG_DIR,
    NMME_DIAG_DIR,
    MODES_VARIABILITY_DIAG_DIR,
)

# Default figure output directory for this HPC environment.
FIGURE_OUTDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)


def figure_filename(*parts, ext="png"):
    """Build consistent, readable lowercase snake_case figure filenames."""
    import re

    clean_parts = ["fig"]
    for part in parts:
        if part is None:
            continue
        text = str(part).strip()
        if not text:
            continue
        text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
        if text:
            clean_parts.append(text)

    suffix = ext.lstrip(".").lower()
    return "_".join(clean_parts) + f".{suffix}"

from esp_lab.utils.dask_util import DaskConfig, get_cluster_client, close_cluster


In [ ]:
import dask
from dask.distributed import (
    wait,
    get_client
)
dask.__version__

## Preprocessing:  Data I/O using Dask

### Create Dask Cluster

In [ ]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local') 

# Clean up resources left by a previous execution of this notebook.
previous_cluster = globals().get("cluster")
try:
    previous_client = get_client()
except ValueError:
    previous_client = None
close_cluster(previous_cluster, previous_client)

# Initialize using the centralized esp_lab utility
dask_cfg = DaskConfig(cluster_type=machine_env, workers=30)
cluster, client = get_cluster_client(dask_cfg)


In [ ]:
cluster

### Read in EAM monthly data; Convert to Seasonal averages (DJF, MAM, JJA, SON)
- Chosen field is returned as a dask array with leading dimensions of Y (initialization year), M (ensemble member), and L (lead season). For example, for November starts, L=1 corresponds to first DJF season.
- "time" which gives prediction verification time (centered time for a given season) is also dimensioned with (Y,L)

In [ ]:
%%time
# -----------------------------------------------------------------------------
# Variable-driven setup + multi-case E3SM hindcast setup
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Directory setup
# -----------------------------------------------------------------------------
# Override the root for a different workspace with ESP_LAB_S2D_DIAG_ROOT.
DIAG_ROOT = S2D_DIAG_ROOT
E3SMLE_OUTDIR = E3SMLE_DIAG_DIR
CESM_SMYLE_OUTDIR = CESM_SMYLE_DIAG_DIR
SMYLE_BENCHMARK_DIR = str(CESM_SMYLE_OUTDIR)
NMME_OUTDIR = NMME_DIAG_DIR
MODES_OUTDIR = MODES_VARIABILITY_DIAG_DIR

# Clean model-first diagnostic layout helpers.
# E3SMLE_DIAG_DIR is kept as a backwards-compatible variable name, but it now
# points to the S2D diagnostic root; each case lives under its cache_tag.
E3SM_DIAG_ROOT = Path(E3SMLE_OUTDIR)

# Change this one line to switch diagnostics:
#   field = "TREFHT"  -> ERA5 tas, model/obs K -> degC
#   field = "TS"      -> ERA5 ts, model/obs K -> degC (land and ocean)
#   field = "PRECT"   -> GPCP PRECT, model m/s -> mm/day
#   field = "PSL"     -> ERA5 psl, model/obs Pa -> hPa
#   field = "SST"     -> HadISST2 sst, E3SM surface ocean temperature, SMYLE TS
field = "PRECT"
#field = "TREFHT"
#field = "TS"
#field = "PSL"
#field = "SST"

def e3sm_case_dir(case_info):
    return E3SM_DIAG_ROOT / case_info["cache_tag"]

def e3sm_leadtime_dir(case_info, stage, *parts):
    path = e3sm_case_dir(case_info) / "leadtime_acc" / str(stage) / analysis_component
    for part in parts:
        path = path / str(part)
    return path

def smyle_leadtime_dir(stage, *parts):
    path = Path(CESM_SMYLE_OUTDIR) / "leadtime_acc" / str(stage) / analysis_component
    for part in parts:
        path = path / str(part)
    return path

def convert_kelvin_to_celsius(da):
    """Convert K to degC while preserving lazy evaluation."""
    da = da - 273.15
    da.attrs["units"] = r"$^\circ$C"
    return da


def convert_precip_mps_to_mmday(da):
    """Convert precipitation from m/s to mm/day while preserving lazy evaluation."""
    da = da * (1000.0 * 86400.0)
    da.attrs["units"] = "mm/day"
    return da


def convert_pa_to_hpa(da):
    """Convert pressure from Pa to hPa while preserving lazy evaluation."""
    da = da * 1.0e-2
    da.attrs["units"] = "hPa"
    return da


def no_unit_conversion(da, units=None):
    """Return a lazy copy and optionally standardize the unit attribute."""
    da = da * 1.0
    if units is not None:
        da.attrs["units"] = units
    return da


VAR_CONFIG = {
    "TREFHT": {
        "long_name": "2-m air temperature",
        "plot_name": "Surface air temperature",
        "obs_name": "ERA5",
        "obs_var": "tas",
        "obs_ys": "1979",
        "obs_ye": "2019",
        "model_convert": convert_kelvin_to_celsius,
        "smyle_convert": convert_kelvin_to_celsius,
        "obs_convert": convert_kelvin_to_celsius,
        "units": r"$^\circ$C",
    },
    "TS": {
        "long_name": "surface radiative temperature",
        "plot_name": "Surface temperature",
        "obs_name": "ERA5",
        "obs_var": "ts",
        "obs_ys": "1979",
        "obs_ye": "2019",
        "model_convert": convert_kelvin_to_celsius,
        "smyle_convert": convert_kelvin_to_celsius,
        "obs_convert": convert_kelvin_to_celsius,
        "units": r"$^\circ$C",
    },
    "PRECT": {
        "long_name": "precipitation",
        "plot_name": "Precipitation",
        "obs_name": "GPCP_v2.3",
        "obs_var": "PRECT",
        "obs_ys": "1979",
        "obs_ye": "2017",
        "model_convert": convert_precip_mps_to_mmday,
        "smyle_convert": convert_precip_mps_to_mmday,
        # GPCP PRECT from this archive is already mm/day.
        "obs_convert": lambda da: no_unit_conversion(da, units="mm/day"),
        "units": "mm/day",
    },
    "PSL": {
        "long_name": "sea-level pressure",
        "plot_name": "Sea-level pressure",
        "obs_name": "ERA5",
        "obs_var": "psl",
        "obs_ys": "1979",
        "obs_ye": "2019",
        "model_convert": convert_pa_to_hpa,
        "smyle_convert": convert_pa_to_hpa,
        "obs_convert": convert_pa_to_hpa,
        "units": "hPa",
    },
    "SST": {
        "long_name": "sea-surface temperature",
        "plot_name": "Sea-surface temperature",
        "component": "ocn",
        "e3sm_realm": "ocn",
        "e3sm_field": "timeMonthly_avg_activeTracers_temperature",
        "smyle_field": "TS",
        "obs_name": "HadISST2",
        "obs_var": "sst",
        "obs_ys": "1979",
        "obs_ye": "2022",
        # E3SM MPAS-O and HadISST2 are already in Celsius; SMYLE TS is Kelvin.
        "model_convert": lambda da: no_unit_conversion(da, units=r"$^\circ$C"),
        "smyle_convert": convert_kelvin_to_celsius,
        "obs_convert": lambda da: no_unit_conversion(da, units=r"$^\circ$C"),
        "units": r"$^\circ$C",
        "ocean_only": True,
    },
}

if field not in VAR_CONFIG:
    raise ValueError(f"Unsupported field={field!r}. Available fields: {list(VAR_CONFIG)}")

cfg = VAR_CONFIG[field]
analysis_component = cfg.get('component', 'atm')
e3sm_field = cfg.get('e3sm_field', field)
smyle_field = cfg.get('smyle_field', field)
print(f"Selected field: {field} ({cfg['long_name']})")
print(f"Observation product: {cfg['obs_name']} variable={cfg['obs_var']}")
print(f"Archive mapping: E3SM {analysis_component}/{e3sm_field}; CESM-SMYLE {smyle_field}")

# Prefer the newer project path, but keep the older location used by the PSL notebook as a fallback.
data_dir_candidates = [
    "/global/cfs/cdirs/e3sm/S2S2D/post_process",
    "/global/cfs/cdirs/e3smdata/simulations/S2S2D/post_process",
]

data_dir = next((p for p in data_dir_candidates if os.path.exists(p)), data_dir_candidates[0])
print(f"Using post-processed data directory: {data_dir}")

# Multi-case E3SM hindcasts. Add more entries here as new post-processed
# hindcasts land under data_dir with the same directory convention.
E3SM_CASES = {
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
    },
#    "E3SM-4DEnVarOcn": {
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "E3SMv3-4DEnVarOcn",
#    },
}

# Which E3SM case should old single-case cells/variables point to.
E3SM_REFERENCE_CASE = "E3SM-FOSIRL"

# Central workflow settings. Edit this block first; later cells consume these
# values so run-control, grid, and chunk choices stay consistent.
WORKFLOW_SETTINGS = {
    "run": {
        "years": (1980, 2018),
        "exclude_year": None,
        "init_months": [5, 11],
        "climatology_years": (1981, 2010),
    },
    "e3sm": {
        "nens": 10,
        "nlead": 24,
        "engine": "netcdf4",
        "realm": "atm",
        "grid": "180x360_aave",
        "freq": "monthly",
        "ts_split": "2yr",
        "require_all_members": True,
        "verify_field_name": True,
        "verify_coverage": True,
        "open_chunks": {},  # e.g., {"L": 24} if open-time chunking is needed.
        "monthly_chunks": {"Y": 3, "L": 24, "M": 2, "lat": 90, "lon": 180},
        "encoding_chunksizes": (1, 8, 1, 90, 180),
    },
    "regrid": {
        "target_dlat": 1.0,
        "target_dlon": 1.0,
        "method": "conservative",
        "periodic": True,
    },
    "smyle": {
        "nens": 20,
        "nlead": 24,
        "chunks": {"Y": 3, "L": -1, "M": 2, "lat": 96, "lon": 144},
    },
    "obs": {
        "verbose": True,
        "chunks": {"time": 24, "lat": 90, "lon": 180},
    },
    "skill": {
        "force_compute": False,
        "detrend": True,
        "model_chunks": {"Y": -1, "L": 8, "M": 2, "lat": 45, "lon": 90},
        "obs_chunks": {"time": -1, "lat": 45, "lon": 90},
    },
    "finite_ensemble_compare": {
        "mode": "final",  # "test" or "final"
        "case_selection": "all",
        "force_recompute": False,
        "detrend": True,
        "random_seed": 42,
        "cache_anomalies": True,
        "persist_inputs": False,
        "model_chunks": {"Y": -1, "M": -1, "L": 1, "lat": 30, "lon": 60},
        "obs_chunks": {"time": -1, "lat": 30, "lon": 60},
        "iteration_batch_size": 10,
        "modes": {
            "test": {"init_months": [11], "iterations": 10, "lead_start": 1, "lead_end": 8},
            "final": {"init_months": [5, 11], "iterations": 100, "lead_start": 1, "lead_end": 8},
        },
    },
    "final_acc_figures": {
        "init_months": [5, 11],
        "max_leads": 7,
        "geometry": {
            "fig_col_width": 4.4,
            "fig_row_height": 2.8,
            "section_gap": 0.04,
            "section_gap_fig_width": 1.2,
            "tight_layout_rect": [0.0, 0.11, 1.0, 0.94],
            "subplots_top": 0.94,
            "subplots_bottom": 0.11,
            "subplots_hspace": 0.06,
            "subplots_wspace": 0.025,
            "month_header_y": 0.955,
            "divider_y0": 0.08,
            "divider_y1": 0.94,
            "divider_color": "0.25",
            "divider_linewidth": 1.0,
            "cbar_rect": [0.3, 0.03, 0.5, 0.015],
        },
        "markers": {
            "sig_level": 0.1,
            "latlim": 80,
            "marker_color": "black",
            "marker_stride": 5,
            "open_marker_size": 12,
            "filled_marker_size": 5,
            "marker_linewidth": 0.65,
            "grid_linewidth": 0.35,
            "grid_color": "0.35",
            "grid_alpha": 0.35,
            "tick_length": 2.5,
            "tick_width": 0.5,
            "label_bbox": {
                "boxstyle": "round,pad=0.2",
                "facecolor": "white",
                "edgecolor": "lightgray",
                "alpha": 0.85,
                "linewidth": 0.8,
            },
        },
        "acc": {
            "include_cesm_smyle": True,
            "cesm_label": "CESM-SMYLE",
            "ci": 0.1,
            "cmin": -1,
            "cmax": 1,
            "cmap": "blue2red_acc",
            "cutoff": 0.5,
            "fontz": 14,
            "title_template": "{plot_name} ACC: CESM-SMYLE and E3SM, linear-detrend",
            "caption": "ACC comparison for CESM-SMYLE and selected E3SM hindcasts, grouped by May/November initialization.",
        },
        "skilldiff": {
            "ci": 0.05,
            "cmin": -0.5,
            "cmax": 0.5,
            "cmap": "blue2red",
            "cutoff": 0.5,
            "fontz": 16,
            "title_template": "{plot_name} ACC difference: E3SM minus matched CESM-SMYLE, linear-detrend",
            "save_title_template": "Leadtime ACC Skill Difference: {field}",
            "caption": "ACC skill difference for selected E3SM hindcasts, grouped by May/November initialization.",
        },
        "font_weight": "bold",
    },
    "diagnostics": {
        # Full drift checks can materialize large arrays; keep them opt-in.
        "run_drift_check": False,
        "drift_check_sample": {"Y": 0, "M": 0, "lat": 0, "lon": 0},
        # The pre-significance ACC overview is useful but large; keep saved
        # output enabled while avoiding an expensive inline notebook render.
        "make_pre_significance_acc_plot": True,
        "pre_significance_acc_dpi": 300,
        "show_pre_significance_acc_plot": False,
    },
    "cache": {
        "force_rewrite_inputs": False,
        "debug_load": False,
        "debug_seasonal_cache": False,
        "cleanup_temp_files": True,
        "temp_file_max_age_hours": 0.0,
    },
}

def _copy_chunks(section, key):
    return dict(WORKFLOW_SETTINGS[section][key])

def cleanup_netcdf_temp_files(path, max_age_hours=0.0):
    """Remove stale temp files left by interrupted atomic NetCDF writes."""
    import time

    path = Path(path)
    if not path.parent.exists():
        return

    cutoff = time.time() - (max_age_hours * 3600.0)
    for tmp_path in path.parent.glob(f".{path.name}.tmp.*"):
        try:
            if max_age_hours <= 0.0 or tmp_path.stat().st_mtime <= cutoff:
                tmp_path.unlink()
        except OSError as exc:
            print(f"Warning: could not remove temp file {tmp_path}: {exc}")

def safe_to_netcdf(ds, path, encoding=None):
    """Write NetCDF atomically so interrupted runs do not corrupt caches."""
    import uuid

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if WORKFLOW_SETTINGS["cache"].get("cleanup_temp_files", True):
        cleanup_netcdf_temp_files(
            path,
            max_age_hours=WORKFLOW_SETTINGS["cache"].get("temp_file_max_age_hours", 0.0),
        )
    tmp_path = path.with_name(f".{path.name}.tmp.{uuid.uuid4().hex}")
    try:
        ds.to_netcdf(tmp_path, encoding=encoding)
        os.replace(tmp_path, path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

years, yeare = WORKFLOW_SETTINGS["run"]["years"]
yexcl = WORKFLOW_SETTINGS["run"]["exclude_year"]
init_months = list(WORKFLOW_SETTINGS["run"]["init_months"])
climy0, climy1 = WORKFLOW_SETTINGS["run"]["climatology_years"]

case_nens = WORKFLOW_SETTINGS["e3sm"]["nens"]
case_nlead = WORKFLOW_SETTINGS["e3sm"]["nlead"]
engine = WORKFLOW_SETTINGS["e3sm"]["engine"]
realm = cfg.get("e3sm_realm", WORKFLOW_SETTINGS["e3sm"]["realm"])
grid = WORKFLOW_SETTINGS["e3sm"]["grid"]
freq = WORKFLOW_SETTINGS["e3sm"]["freq"]
ts_split = WORKFLOW_SETTINGS["e3sm"]["ts_split"]
require_all_members = WORKFLOW_SETTINGS["e3sm"]["require_all_members"]
verify_field_name = WORKFLOW_SETTINGS["e3sm"]["verify_field_name"]
verify_coverage = WORKFLOW_SETTINGS["e3sm"]["verify_coverage"]
chunks_open = _copy_chunks("e3sm", "open_chunks")
mchunk = _copy_chunks("e3sm", "monthly_chunks")
debug = WORKFLOW_SETTINGS["cache"]["debug_load"]

lead_years = [y for y in np.arange(years, yeare + 1) if y != yexcl]
INIT_YEARS_BY_MONTH = {init_month: lead_years for init_month in init_months}
members = [f"EN{i:02d}" for i in range(case_nens)]

e3sm_raw_by_case_month = {}

for case_key, case_info in E3SM_CASES.items():
    case_prefix = case_info["case_prefix"]
    e3sm_raw_by_case_month[case_key] = {}

    print("")
    print("=" * 80)
    print(f"Loading {case_key}: {case_prefix}")

    for init_month in init_months:
        init_tags = data_access.build_init_tags(INIT_YEARS_BY_MONTH[init_month], init_month)

        e3sm = data_access.get_monthly_data(
            data_dir=data_dir,
            case_prefix=case_prefix,
            members=members,
            init_tags=init_tags,
            field=e3sm_field,
            nlead=case_nlead,
            chunks=chunks_open,
            realm=realm,
            grid=grid,
            freq=freq,
            ts_split=ts_split,
            require_all_members=require_all_members,
            verify_field_name=verify_field_name,
            verify_coverage=verify_coverage,
            engine=engine,
        )

        if e3sm_field != field:
            e3sm = e3sm.rename({e3sm_field: field})

        print(f"{case_key} init_month={init_month}, size={e3sm.nbytes / 1e9:.2f} GB")

        # Apply final chunking after combine.
        e3sm = e3sm.chunk(mchunk)

        if debug:
            print(e3sm)
            print(e3sm.dims)
            print(e3sm.Y.values)
            print(e3sm.L.values[:5], e3sm.L.values[-5:])
            print(e3sm.time.isel(Y=0, L=slice(0, 3)).values)

        e3sm_raw_by_case_month[case_key][init_month] = e3sm

# Backward-compatible aliases for older exploratory cells.
e3smle_by_month = e3sm_raw_by_case_month[E3SM_REFERENCE_CASE]
e3smle05 = e3smle_by_month.get(5)
e3smle11 = e3smle_by_month.get(11)



### Store datasets to disk for quicker processing next time (note this takes >30 minutes)

In [ ]:
%%time
# Store seasonal E3SM datasets to disk for quicker processing next time.
# Files are stored under <case>/leadtime_acc/inputs/<component>/<field>/.

debug = WORKFLOW_SETTINGS["cache"]["debug_seasonal_cache"]
force_rewrite = WORKFLOW_SETTINGS["cache"]["force_rewrite_inputs"]
encoding = {
    field: {
        "chunksizes": WORKFLOW_SETTINGS["e3sm"]["encoding_chunksizes"],
        "zlib": True,
        "complevel": 1
    }
}

e3sm_seas_by_case_month = {}

for case_key, case_info in E3SM_CASES.items():
    cache_tag = case_info["cache_tag"]
    input_dir = e3sm_leadtime_dir(case_info, "inputs", field)
    input_dir.mkdir(parents=True, exist_ok=True)
    e3sm_seas_by_case_month[case_key] = {}

    for init_month in init_months:
        outname = f"{cache_tag}_{init_month:02d}_{field}_N{case_nens:02d}_M{case_nlead:02d}_seas.nc"
        outfile = input_dir / outname

        if os.path.exists(outfile) and not force_rewrite:
            e3sm_seas = xr.open_dataset(outfile, chunks=mchunk)
        else:
            e3sm = e3sm_raw_by_case_month[case_key][init_month]

            # 1. seasonal aggregation (lazy)
            e3sm_seas = cal.mon_to_seas_dask(e3sm)

            # 2. rechunk after rolling
            e3sm_seas = e3sm_seas.chunk(mchunk)

            # 3. persist (optional but OK since reused)
            e3sm_seas = e3sm_seas.persist()

            # 4. write with encoding
            print(f"Writing {outfile}")
            safe_to_netcdf(e3sm_seas, outfile, encoding=encoding)

            # 5. reopen from disk to keep graph small
            e3sm_seas = xr.open_dataset(outfile, chunks=mchunk)

        e3sm_seas_by_case_month[case_key][init_month] = e3sm_seas

        if debug:
            print(f"\n{case_key} init_month={init_month}")
            print(e3sm_seas)
            print(e3sm_seas.dims)
            print(e3sm_seas.Y.values)
            print(e3sm_seas.L.values[:5], e3sm_seas.L.values[-5:])
            print(e3sm_seas.time.isel(Y=0, L=slice(0, 3)).values)

# Backward-compatible aliases.
e3smle_seas_by_month = e3sm_seas_by_case_month[E3SM_REFERENCE_CASE]
e3smle05_seas = e3smle_seas_by_month.get(5)
e3smle11_seas = e3smle_seas_by_month.get(11)

if debug:
    print("available seasonal E3SM cases:", sorted(e3sm_seas_by_case_month))
    print("available seasonal init months:", sorted(e3smle_seas_by_month))



### Regrid Hindcast data
- Regrid all hindcast outputs onto a common regular latitude–longitude grid (e.g., 5°×5°) to ensure consistent comparison across datasets
- Support multiple source grid types: a. Unstructured grids (e.g., E3SM/CAM-SE ncol) and b. tructured lat–lon grids (e.g., CESM/CAM-FV or post-processed E3SM)
- Apply a two-step workflow for unstructured grids: 1. Remap from native ncol to structured lat–lon using precomputed sparse mapping weights and 2. Regrid from intermediate lat–lon to target grid

In [ ]:
%%time
# Regrid all E3SM hindcast data to the common analysis grid.
target_dlat = WORKFLOW_SETTINGS["regrid"]["target_dlat"]
target_dlon = WORKFLOW_SETTINGS["regrid"]["target_dlon"]

destgrid = regrid.make_latlon_grid(dlat=target_dlat, dlon=target_dlon)
regrid_method = WORKFLOW_SETTINGS["regrid"]["method"]
regrid_periodic = WORKFLOW_SETTINGS["regrid"]["periodic"]

# Ensure clean chunking before regrid.
e3sm_seas_by_case_month = {
    case_key: {
        init_month: ds.chunk(mchunk)
        for init_month, ds in by_month.items()
    }
    for case_key, by_month in e3sm_seas_by_case_month.items()
}

# Build one regridder from the reference E3SM case/month.
_ref_month = init_months[0]
regridder = regrid.make_regridder(
    e3sm_seas_by_case_month[E3SM_REFERENCE_CASE][_ref_month],
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

# Regrid, rechunk, and apply variable-specific unit conversion by case/init month.
e3sm_da_by_case_month = {}
for case_key in E3SM_CASES:
    e3sm_da_by_case_month[case_key] = {}

    for init_month in init_months:
        da = regridder(e3sm_seas_by_case_month[case_key][init_month][field]).chunk(mchunk)
        da = cfg["model_convert"](da)
        e3sm_da_by_case_month[case_key][init_month] = da

        print(f"{case_key} init_month={init_month}")
        print(da.shape)
        print(da.dims)
        print(da.Y.values)
        print(da.L.values[:5], da.L.values[-5:])
        print("units:", da.attrs.get("units"))

# Backward-compatible convenience variables for exploratory cells.
e3smle_da_by_month = e3sm_da_by_case_month[E3SM_REFERENCE_CASE]
e3smle05_da = e3smle_da_by_month.get(5)
e3smle11_da = e3smle_da_by_month.get(11)



### Load CESM-SMYLE benchmark data
- Benchmark files are pre-generated once by `scripts/run_process_cesm_smyle_benchmark.py` and stored under `<S2D_DIAG_ROOT>/CESM-SMYLE/leadtime_acc/inputs/atm/<native-field>/`; logical SST uses the native CESM-SMYLE field `TS`.
- Each file contains seasonal means on the native f09_g17 grid with dimensions `(Y, L, M, lat, lon)`.
- The benchmark is regridded here to the same analysis grid as E3SM before skill is computed.


In [ ]:
%%time
# Load and regrid CESM-SMYLE benchmark data to the same analysis grid as E3SM.
from esp_lab import data_access_cesm_smyle as smyle_access

SMYLE_BENCHMARK_DIR = str(CESM_SMYLE_OUTDIR)
smyle_nens = WORKFLOW_SETTINGS["smyle"]["nens"]
smyle_nlead = WORKFLOW_SETTINGS["smyle"]["nlead"]
smyle_skill_cache_version = smyle_access.VERIFICATION_TIME_VERSION
mchunk_smyle = _copy_chunks("smyle", "chunks")

smyle_seas_by_month = {}
for init_month in init_months:
    ds = smyle_access.load_benchmark(
        field=smyle_field,
        init_month=init_month,
        benchmark_dir=SMYLE_BENCHMARK_DIR,
        nens=smyle_nens,
        nlead=smyle_nlead,
        freq="seas",
        chunks=mchunk_smyle,
    )
    smyle_seas_by_month[init_month] = ds.chunk(mchunk_smyle)
    print(f"CESM-SMYLE init_month={init_month}: {ds.sizes}")

regridder_smyle = regrid.make_regridder(
    smyle_seas_by_month[init_months[0]],
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

smyle_da_by_month = {}
for init_month in init_months:
    da = regridder_smyle(smyle_seas_by_month[init_month][smyle_field]).chunk(mchunk)
    da = cfg["smyle_convert"](da)
    smyle_da_by_month[init_month] = da

    print(f"CESM-SMYLE regridded init_month={init_month}: {da.sizes}")
    print("units:", da.attrs.get("units"))

smyle05_seas = smyle_seas_by_month.get(5)
smyle11_seas = smyle_seas_by_month.get(11)
smyle05_da = smyle_da_by_month.get(5)
smyle11_da = smyle_da_by_month.get(11)


In [ ]:
for case_key, by_month in e3sm_da_by_case_month.items():
    for init_month, da in by_month.items():
        da.isel(Y=0, L=0, M=0).plot()
        plt.title(f"{case_key}, init_month={init_month}")
        plt.show()



### Get Observational datasets
- Load observational datasets from the E3SM Diags archive (CMOR variables).
- Harmonize in time, variable naming, and space (regridding to a common grid).
- Provide consistent reference fields for hindcast evaluation.
- Compute OBS seasonal averages as a simple rolling mean
- Regrid to analysis grid 

In [ ]:
%%time
# Load observations using the selected field configuration.
obs_dir = "/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series"
obs_name = cfg["obs_name"]
obs_var = cfg["obs_var"]
obs_map = {field: obs_var}
obs_ys = cfg["obs_ys"]
obs_ye = cfg["obs_ye"]
verbose = WORKFLOW_SETTINGS["obs"]["verbose"]
obs_chunks = _copy_chunks("obs", "chunks")

obs_monthly = obs_access.get_monthly_data(
    obs_dir=obs_dir,
    field=field,
    field_map=obs_map,
    product=obs_name,
    start_year=obs_ys,
    end_year=obs_ye,
    chunks=obs_chunks,
    verbose=verbose,
)

# Compute OBS seasonal averages.
obs_seas = obs_access.mon_to_seas_obs(
    obs_monthly,
    var=obs_var,
    field_map=obs_map,
)

# Rechunk after rolling.
obs_seas = obs_seas.chunk(obs_chunks)

# Build regridder from Dataset, not DataArray.
regridder_obs = regrid.make_regridder(
    obs_monthly,
    destgrid,
    method=regrid_method,
    periodic=regrid_periodic,
)

# Regrid seasonal DataArray.
try:
    obs_seas_rg = regridder_obs(
        obs_seas,
        output_chunks=obs_chunks,
    )
except TypeError:
    obs_seas_rg = regridder_obs(obs_seas)

# Rechunk after regrid and apply variable-specific unit conversion.
obs_seas_rg = obs_seas_rg.chunk(obs_chunks)
print(obs_seas_rg)

obs_seas_rg = cfg["obs_convert"](obs_seas_rg)

# HadISST2 SST is filled over land, so its non-null values cannot define
# an ocean mask. For SST, derive the static common domain from valid
# E3SM MPAS-O surface-temperature cells, then apply it consistently to
# HadISST2, E3SM, and CESM-SMYLE before computing anomalies and skill.
if cfg.get("ocean_only", False):
    e3sm_ocean_masks = []
    for case_key in E3SM_CASES:
        for init_month in init_months:
            sample = e3sm_da_by_case_month[case_key][init_month].isel(
                Y=0, M=0, L=0, drop=True
            )
            e3sm_ocean_masks.append(sample.notnull())
    ocean_mask = (
        xr.concat(e3sm_ocean_masks, dim="mask_source")
        .all("mask_source")
        .rename("ocean_mask")
        .compute()
    )
    obs_seas_rg = obs_seas_rg.where(ocean_mask)
    for case_key in E3SM_CASES:
        for init_month in init_months:
            e3sm_da_by_case_month[case_key][init_month] = (
                e3sm_da_by_case_month[case_key][init_month].where(ocean_mask)
            )
    for init_month in init_months:
        smyle_da_by_month[init_month] = smyle_da_by_month[init_month].where(ocean_mask)
    obs_seas_rg.attrs["analysis_domain"] = "common valid E3SM MPAS-O surface-ocean cells"
    obs_seas_rg.attrs["ocean_mask_source"] = "E3SM MPAS-O SST finite-value mask"
    print(
        f"Applied E3SM MPAS-O ocean mask to observations and both model families "
        f"({int(ocean_mask.sum().item())}/{ocean_mask.size} target-grid cells retained)."
    )
print(obs_seas_rg)

# Backward-compatible aliases for older exploratory cells.
if cfg["obs_name"] == "ERA5":
    obs_era5_seas_rg = obs_seas_rg
if field == "PRECT":
    obs_gpcp_seas_rg = obs_seas_rg


In [ ]:
obs_seas_rg.isel(time=100).plot()

# Skill Analysis

In [ ]:
%%time
time_by_case_month = {
    case_key: {
        init_month: e3sm_seas_by_case_month[case_key][init_month].time.load()
        for init_month in init_months
    }
    for case_key in E3SM_CASES
}

# Backward-compatible convenience variables for exploratory cells.
time_by_month = time_by_case_month[E3SM_REFERENCE_CASE]
e3smle05_time = time_by_month.get(5)
e3smle11_time = time_by_month.get(11)



In [ ]:
%%time
# Compute de-drifted anomalies from specified climatology.

anom_by_case_month = {}
clim_by_case_month = {}

for case_key in E3SM_CASES:
    anom_by_case_month[case_key] = {}
    clim_by_case_month[case_key] = {}

    for init_month in init_months:
        anom_by_case_month[case_key][init_month], clim_by_case_month[case_key][init_month] = stats.remove_drift(
            e3sm_da_by_case_month[case_key][init_month],
            time_by_case_month[case_key][init_month],
            climy0,
            climy1,
        )

# Backward-compatible convenience variables for exploratory cells.
anom_by_month = anom_by_case_month[E3SM_REFERENCE_CASE]
clim_by_month = clim_by_case_month[E3SM_REFERENCE_CASE]
e3smle05_anom = anom_by_month.get(5)
e3smle11_anom = anom_by_month.get(11)
e3smle05_clim = clim_by_month.get(5)
e3smle11_clim = clim_by_month.get(11)



In [ ]:
%%time
# Compute CESM-SMYLE benchmark de-drifted anomalies from the same climatology window.
smyle_time_by_month = {
    init_month: smyle_seas_by_month[init_month].time.load()
    for init_month in init_months
}

smyle_anom_by_month = {}
smyle_clim_by_month = {}

for init_month in init_months:
    smyle_anom_by_month[init_month], smyle_clim_by_month[init_month] = stats.remove_drift(
        smyle_da_by_month[init_month],
        smyle_time_by_month[init_month],
        climy0,
        climy1,
    )

smyle05_time = smyle_time_by_month.get(5)
smyle11_time = smyle_time_by_month.get(11)
smyle05_anom = smyle_anom_by_month.get(5)
smyle11_anom = smyle_anom_by_month.get(11)
smyle05_clim = smyle_clim_by_month.get(5)
smyle11_clim = smyle_clim_by_month.get(11)


In [ ]:
# Optional lightweight sanity check for drift removal.
# The full anomaly arrays are large, so this cell is disabled by default.

def check_remove_drift_sample(da, da_time, anom, clim, climy0, climy1, sample=None, name="case"):
    d1 = cftime.DatetimeNoLeap(climy0, 1, 1, 0, 0, 0)
    d2 = cftime.DatetimeNoLeap(climy1, 12, 31, 23, 59, 59)
    sample = dict(sample or {})

    # Check dimensions lazily first.
    print(f"=== {name} ===")
    print("raw dims :", da.dims, da.shape)
    print("anom dims:", anom.dims, anom.shape)
    print("clim dims:", clim.dims, clim.shape)

    # Then test one configurable point across Y/M/L rather than materializing
    # the full spatial array for every case and start month.
    point = {
        dim: min(int(sample.get(dim, 0)), da.sizes[dim] - 1)
        for dim in sample
        if dim in da.dims and da.sizes[dim] > 0
    }
    da_point = da.isel(point)
    anom_point = anom.isel({dim: idx for dim, idx in point.items() if dim in anom.dims})
    clim_point = clim.isel({dim: idx for dim, idx in point.items() if dim in clim.dims})
    time_point = da_time.isel({dim: idx for dim, idx in point.items() if dim in da_time.dims})

    masked = da_point.where((time_point >= d1) & (time_point <= d2))
    clim_manual = masked.mean("M").mean("Y")
    anom_manual = da_point - clim_manual
    masked_anom = anom_point.where((time_point >= d1) & (time_point <= d2))

    xr.testing.assert_allclose(clim_point.load(), clim_manual.load())
    xr.testing.assert_allclose(anom_point.load(), anom_manual.load())
    print(f"sample reconstruction PASS: {point}")

    resid = masked_anom.mean(("Y", "M")).load()
    print("sample max abs masked anomaly mean:", float(abs(resid).max(skipna=True)))


if WORKFLOW_SETTINGS["diagnostics"].get("run_drift_check", False):
    drift_check_sample = WORKFLOW_SETTINGS["diagnostics"].get("drift_check_sample", {})
    for case_key in E3SM_CASES:
        for init_month in init_months:
            check_remove_drift_sample(
                e3sm_da_by_case_month[case_key][init_month],
                time_by_case_month[case_key][init_month],
                anom_by_case_month[case_key][init_month],
                clim_by_case_month[case_key][init_month],
                climy0,
                climy1,
                drift_check_sample,
                f"{case_key}, init_month={init_month}",
            )
else:
    print("Skipping optional drift-removal diagnostic; set WORKFLOW_SETTINGS['diagnostics']['run_drift_check'] = True to run a sampled check.")


In [ ]:
%%time
# Compute and save E3SM skill data separately for each case/init month.
# Files are stored under <case>/leadtime_acc/skill/<component>/<field>/.
e3smle_skill_dir = None

# False reuses existing skill NetCDF files; set True only for intentional regeneration.
force_compute = WORKFLOW_SETTINGS["skill"]["force_compute"]
detrend = WORKFLOW_SETTINGS["skill"]["detrend"]
skill_chunk_model = _copy_chunks("skill", "model_chunks")
skill_chunk_obs = _copy_chunks("skill", "obs_chunks")

clim_start = cftime.DatetimeNoLeap(climy0, 1, 1)
clim_end = cftime.DatetimeNoLeap(climy1, 12, 31)

skill_by_case_month = {}
obs_skill = obs_seas_rg.chunk(skill_chunk_obs)

for case_key, case_info in E3SM_CASES.items():
    cache_tag = case_info["cache_tag"]
    skill_dir = e3sm_leadtime_dir(case_info, "skill", field)
    skill_dir.mkdir(parents=True, exist_ok=True)
    skill_by_case_month[case_key] = {}

    for init_month in init_months:
        if detrend:
            outname = f"{cache_tag}{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_detrend.nc"
        else:
            outname = f"{cache_tag}{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}.nc"

        outfile = skill_dir / outname

        if os.path.exists(outfile) and not force_compute:
            skill_ds = xr.open_dataset(outfile).load()
        else:
            model_anom = anom_by_case_month[case_key][init_month].chunk(skill_chunk_model)
            model_time = time_by_case_month[case_key][init_month]

            skill = stats.compute_skill_seasonal(
                model_anom,
                model_time,
                obs_skill,
                clim_start,
                clim_end,
                1,
                8,
                resamp=0,
                detrend=detrend,
            )

            skill_ds = xr.Dataset({
                "corr": skill.corr,
                "pval": skill.pval,
                "rmse": skill.rmse,
                "msss": skill.msss,
                "rpc": skill.rpc,
                "sig_obs": skill.sig_obs,
                "sig_sig": skill.sig_sig,
                "sig_tot": skill.sig_tot,
                "s2t": skill.s2t,
            }).compute()

            safe_to_netcdf(skill_ds, outfile)

        skill_by_case_month[case_key][init_month] = skill_ds

        print(outfile)
        print(skill_ds)

# Backward-compatible aliases.
skill_by_month = skill_by_case_month[E3SM_REFERENCE_CASE]
skill05_ds = skill_by_month.get(5)
skill11_ds = skill_by_month.get(11)



In [ ]:
%%time
# Compute and save CESM-SMYLE benchmark skill with the same observations/settings.
# Files are stored under CESM-SMYLE/leadtime_acc/skill/<component>/<field>/.
smyle_skill_outdir = smyle_leadtime_dir("skill", field)
os.makedirs(smyle_skill_outdir, exist_ok=True)

smyle_skill_by_month = {}
smyle_overlap_skill_by_month = {}
obs_skill = obs_skill.chunk(skill_chunk_obs)

common_years = np.intersect1d(
    smyle_anom_by_month[init_months[0]].Y.values,
    e3sm_raw_by_case_month[E3SM_REFERENCE_CASE][init_months[0]].Y.values,
)

for init_month in init_months:
    if detrend:
        outname = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_detrend_{smyle_skill_cache_version}.nc"
        outname_overlap = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_overlap_detrend_{smyle_skill_cache_version}.nc"
    else:
        outname = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_{smyle_skill_cache_version}.nc"
        outname_overlap = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_overlap_{smyle_skill_cache_version}.nc"

    outfile = os.path.join(smyle_skill_outdir, outname)
    outfile_overlap = os.path.join(smyle_skill_outdir, outname_overlap)

    # 1. Full 50-year CESM-SMYLE skill
    if os.path.exists(outfile) and not force_compute:
        skill_ds = xr.open_dataset(outfile).load()
    else:
        model_anom = smyle_anom_by_month[init_month].chunk(skill_chunk_model)
        model_time = smyle_time_by_month[init_month]

        skill = stats.compute_skill_seasonal(
            model_anom,
            model_time,
            obs_skill,
            clim_start,
            clim_end,
            1,
            8,
            resamp=0,
            detrend=detrend,
        )

        skill_ds = xr.Dataset({
            "corr": skill.corr,
            "pval": skill.pval,
            "rmse": skill.rmse,
            "msss": skill.msss,
            "rpc": skill.rpc,
            "sig_obs": skill.sig_obs,
            "sig_sig": skill.sig_sig,
            "sig_tot": skill.sig_tot,
            "s2t": skill.s2t,
        }).compute()

        safe_to_netcdf(skill_ds, outfile)

    smyle_skill_by_month[init_month] = skill_ds

    # 2. Common overlap years (39-year) CESM-SMYLE skill
    if os.path.exists(outfile_overlap) and not force_compute:
        skill_ds_overlap = xr.open_dataset(outfile_overlap).load()
    else:
        model_anom_ov = smyle_anom_by_month[init_month].sel(Y=common_years).chunk(skill_chunk_model)
        model_time_ov = smyle_time_by_month[init_month].sel(Y=common_years)

        skill_ov = stats.compute_skill_seasonal(
            model_anom_ov,
            model_time_ov,
            obs_skill,
            clim_start,
            clim_end,
            1,
            8,
            resamp=0,
            detrend=detrend,
        )

        skill_ds_overlap = xr.Dataset({
            "corr": skill_ov.corr,
            "pval": skill_ov.pval,
            "rmse": skill_ov.rmse,
            "msss": skill_ov.msss,
            "rpc": skill_ov.rpc,
            "sig_obs": skill_ov.sig_obs,
            "sig_sig": skill_ov.sig_sig,
            "sig_tot": skill_ov.sig_tot,
            "s2t": skill_ov.s2t,
        }).compute()

        safe_to_netcdf(skill_ds_overlap, outfile_overlap)

    smyle_overlap_skill_by_month[init_month] = skill_ds_overlap

smyle05_skill_ds = smyle_skill_by_month.get(5)
smyle11_skill_ds = smyle_skill_by_month.get(11)

skill_delta_by_case_month = {
    case_key: {
        init_month: skill_by_case_month[case_key][init_month].corr - smyle_overlap_skill_by_month[init_month].corr
        for init_month in init_months
    }
    for case_key in E3SM_CASES
}

skill_delta_by_month = skill_delta_by_case_month[E3SM_REFERENCE_CASE]



### Final ACC Plot: BSMYLE vs E3SMLE (Common Overlap Years)
- Reads saved skill files for both BSMYLE/CESM-SMYLE and E3SMLE evaluated over the **common overlap years (1980–2018)**.
- Plots each initialization month as paired columns: CESM-SMYLE on the left, E3SMLE on the right.
- Ensures 100% baseline ACC shading alignment with the subsequent significance figure.



In [ ]:
%%time
# Read saved CESM-SMYLE and multi-case E3SM skill data and plot side by side.
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from matplotlib.offsetbox import AnchoredText

from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import mov_utils as mov

make_pre_significance_acc_plot = WORKFLOW_SETTINGS.get("diagnostics", {}).get(
    "make_pre_significance_acc_plot",
    False,
)

if not make_pre_significance_acc_plot:
    print(
        "Skipping pre-significance ACC overview plot; set "
        "WORKFLOW_SETTINGS['diagnostics']['make_pre_significance_acc_plot'] = True to run it."
    )
else:
    # This plot cell intentionally consumes the single field/configuration selected
    # in the variable-driven setup cell above. Do not silently fall back to another
    # field here; that can mix cached products from different diagnostics.
    _required_setup_names = [
        "field",
        "cfg",
        "init_months",
        "climy0",
        "climy1",
        "detrend",
        "E3SM_CASES",
        "E3SM_REFERENCE_CASE",
        "smyle_leadtime_dir",
        "smyle_skill_cache_version",
        "e3sm_leadtime_dir",
    ]
    _missing_setup_names = [name for name in _required_setup_names if name not in globals()]
    if _missing_setup_names:
        raise RuntimeError(
            "Run the variable-driven setup cell before this final ACC plotting cell; "
            f"missing: {_missing_setup_names}"
        )
    FIGURE_OUTDIR = globals().get(
        "FIGURE_OUTDIR",
        Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag"),
    )
    FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

    if "figure_filename" not in globals():
        def figure_filename(*parts, ext="png"):
            clean = ["fig"]
            for part in parts:
                token = re.sub(r"[^A-Za-z0-9]+", "_", str(part)).strip("_").lower()
                if token:
                    clean.append(token)
            return "_".join(clean) + f".{ext.lstrip('.').lower()}"

    # ---------------------------------------------------------------------------
    # User-adjustable final-plot setup
    # ---------------------------------------------------------------------------
    # Import paths if not already available
    if "E3SMLE_OUTDIR" not in globals() or "CESM_SMYLE_OUTDIR" not in globals():
        from esp_lab.paths import (
            E3SMLE_DIAG_DIR,
            CESM_SMYLE_DIAG_DIR,
        )
        E3SMLE_OUTDIR = globals().get("E3SMLE_OUTDIR", E3SMLE_DIAG_DIR)
        CESM_SMYLE_OUTDIR = globals().get("CESM_SMYLE_OUTDIR", CESM_SMYLE_DIAG_DIR)
        E3SM_DIAG_ROOT = Path(E3SMLE_OUTDIR)
    else:
        E3SMLE_OUTDIR = globals().get("E3SMLE_OUTDIR")
        CESM_SMYLE_OUTDIR = globals().get("CESM_SMYLE_OUTDIR")

    e3smle_skill_dir = None
    smyle_skill_dir = str(smyle_leadtime_dir("skill", field))

    metric = "ACC"

    model_order = ["CESM-SMYLE"] + list(E3SM_CASES)
    model_display_name = {"CESM-SMYLE": "CESM-SMYLE"}
    model_display_name.update({
        case_key: case_info.get("display_name", case_key)
        for case_key, case_info in E3SM_CASES.items()
    })
    # Seasonal lead denotes the first monthly lead in each 3-month window:
    # lead 1 = months 1-3, lead 4 = months 4-6, etc.
    skill_leads = [1, 4, 7, 10, 13, 16, 19, 22]
    map_plot_leads = skill_leads[:-1]
    season_names = ["DJF", "MAM", "JJA", "SON"]
    init_month_names = {
        1: "JAN", 2: "FEB", 3: "MAR", 4: "APR", 5: "MAY", 6: "JUN",
        7: "JUL", 8: "AUG", 9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC",
    }

    sigon = True
    siglvl = 0.1
    mask_negacc = False

    # Single font-size control for the full multi-panel figure.
    fontz = 20
    panel_title_size = fontz
    lead_label_size = fontz * 0.85
    lat_lon_label_size = fontz * 0.8
    colorbar_label_size = fontz
    colorbar_tick_size = fontz * 0.85
    suptitle_size = fontz * 1.15
    fweight = 'bold'

    # Plot style/layout controls.
    figfmt = "png"
    plot_diag_cfg = WORKFLOW_SETTINGS.get("diagnostics", {})
    fig_col_width = 5.0
    fig_row_height = 3.0
    projection = ccrs.PlateCarree()
    suptitle_y = 0.995
    tight_layout_rect = [0, 0.16, 1, 0.985]
    bottom = 0.10
    wspace = 0.05
    save_bbox = "tight"

    cmap = "blue2red_acc"
    coff = 0.5
    ci = 0.1
    cmin = -1
    cmax = 1

    colorbar_axes = [0.25, 0.055, 0.5, 0.02]
    colorbar_orientation = 'horizontal'
    dpi = plot_diag_cfg.get("pre_significance_acc_dpi", 150)
    show_figure_inline = plot_diag_cfg.get("show_pre_significance_acc_plot", False)

    lon_ticks = [-160, -80, 0, 80, 160]
    lat_ticks = [-60, -30, 0, 30, 60]
    lat_lon_tick_length = 2.5
    lat_lon_tick_width = 0.5
    map_axis_linewidth = 1.0
    gridline_width = 0.35
    gridline_color = "0.35"
    gridline_alpha = 0.35
    gridline_style = "-"

    lead_label_loc = "lower right"
    lead_label_pad = 0.25
    lead_label_borderpad = 0.35
    lead_label_bbox = dict(facecolor=(1, 1, 1, 0.72), edgecolor='black', linewidth=0.3, boxstyle='round,pad=0.1')

    skill_by_model = {model: {} for model in model_order}

    for init_month in init_months:
        if detrend:
            smyle_name = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_overlap_detrend_{smyle_skill_cache_version}.nc"
        else:
            smyle_name = f"BSMYLE{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_overlap_{smyle_skill_cache_version}.nc"

        skill_by_model["CESM-SMYLE"][init_month] = xr.open_dataset(
            os.path.join(smyle_skill_dir, smyle_name)
        ).load()

        for case_key, case_info in E3SM_CASES.items():
            cache_tag = case_info.get("cache_tag", case_key.replace("E3SM-", ""))
            if detrend:
                e3sm_name = f"{cache_tag}{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}_detrend.nc"
            else:
                e3sm_name = f"{cache_tag}{init_month:02d}_{field}_skill_clim_{climy0}_{climy1}.nc"

            e3sm_path = e3sm_leadtime_dir(case_info, "skill", field) / e3sm_name
            skill_by_model[case_key][init_month] = xr.open_dataset(e3sm_path).load()

    # Keep the old variable names available for exploratory cells below/above.
    skill_by_case_month = {
        case_key: skill_by_model[case_key]
        for case_key in E3SM_CASES
    }
    skill_by_month = skill_by_model[E3SM_REFERENCE_CASE]
    smyle_skill_by_month = skill_by_model["CESM-SMYLE"]

    def seasonal_label(init_month, lead):
        center_month = ((init_month + lead + 1 - 1) % 12) + 1
        return season_names[((center_month % 12) // 3)]


    def lead_label(init_month, lead):
        return f"{lead:2d}: {seasonal_label(init_month, lead)}"


    def add_lead_label(ax, text):
        label = AnchoredText(
            text,
            loc=lead_label_loc,
            prop=dict(size=lead_label_size, weight=fweight, family="monospace"),
            frameon=True,
            pad=lead_label_pad,
            borderpad=lead_label_borderpad,
        )
        label.patch.set(**lead_label_bbox)
        ax.add_artist(label)


    def set_map_axis_linewidth(ax):
        for spine in ax.spines.values():
            spine.set_linewidth(map_axis_linewidth)
        if hasattr(ax, "outline_patch"):
            ax.outline_patch.set_linewidth(map_axis_linewidth)


    def add_lat_lon_labels(ax, row, col, nrows, ncols):
        # Keep labels on the outer edges so the multi-panel plot stays readable.
        set_map_axis_linewidth(ax)
        ax.set_xticks(lon_ticks, crs=projection)
        ax.set_yticks(lat_ticks, crs=projection)
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(
            labelsize=lat_lon_label_size,
            length=lat_lon_tick_length,
            width=lat_lon_tick_width,
            top=False,
            right=False,
            labelbottom=(row == nrows - 1),
            labelleft=(col == 0),
        )
        ax.gridlines(
            crs=projection,
            linewidth=gridline_width,
            color=gridline_color,
            alpha=gridline_alpha,
            linestyle=gridline_style,
            draw_labels=False,
        )

    skill_plot_by_model = {
        model: {
            init_month: ds.assign_coords(L=skill_leads[:ds.sizes["L"]])
            for init_month, ds in by_month.items()
        }
        for model, by_month in skill_by_model.items()
    }

    plot_months = [
        init_month for init_month in init_months
        if all(init_month in skill_plot_by_model[model] for model in model_order)
    ]
    plot_leads = [
        lead for lead in map_plot_leads
        if all(
            lead in skill_plot_by_model[model][init_month].L.values
            for model in model_order
            for init_month in plot_months
        )
    ]

    nrows = len(plot_leads)
    ncols = len(plot_months) * len(model_order)

    if detrend:
        subtitle = f"{cfg['plot_name']} skill (1980–2018 overlap years), linear-detrend"
        trendstr = "wt_detrend"
    else:
        subtitle = f"{cfg['plot_name']} skill (1980–2018 overlap years), no-detrend"
        trendstr = "wo_detrend"

    corr_by_model = {}
    for model in model_order:
        corr_by_model[model] = {}
        for init_month in plot_months:
            skill_plot = skill_plot_by_model[model][init_month]
            if sigon:
                corr = skill_plot.corr.where(skill_plot.pval < siglvl)
                sigstr = "wt_sigmask"
            else:
                corr = skill_plot.corr
                sigstr = "wo_sigmask"

            if mask_negacc:
                corr = corr.where(corr >= 0)

            corr_by_model[model][init_month] = corr

    print("=== Sanity check: data range ===")
    for model in model_order:
        for init_month, corr in corr_by_model[model].items():
            print(
                f"{model} init_month={init_month} corr min/max:",
                float(corr.min(skipna=True)),
                float(corr.max(skipna=True)),
            )

    figname = figure_filename(field, "multi_e3sm", "acc_skill_map", ext=figfmt)

    print(f"working on figure: {figname}")

    fig = plt.figure(figsize=(fig_col_width * ncols, fig_row_height * nrows))
    fig.suptitle(subtitle, fontsize=suptitle_size, fontweight=fweight, y=suptitle_y)
    proj = projection
    cntr = None

    for i, lead in enumerate(plot_leads):
        for j, init_month in enumerate(plot_months):
            for k, model in enumerate(model_order):
                skill_plot = skill_plot_by_model[model][init_month]
                subplot = i * ncols + j * len(model_order) + k + 1
                model_name = model_display_name.get(model, model)
                title = (
                    f"{model_name} ({init_month_names.get(init_month, f'{init_month:02d}')})"
                    if i == 0 else ''
                )
                ax, cntr = maps.map_pcolor_global_subplot(
                    fig,
                    corr_by_model[model][init_month].sel(L=lead),
                    skill_plot.lon,
                    skill_plot.lat,
                    ci, cmin, cmax,
                    title,
                    nrows, ncols, subplot,
                    proj, cmap=cmap, cutoff=coff,
                    fontsize=panel_title_size,
                )
                add_lead_label(ax, lead_label(init_month, lead))
                add_lat_lon_labels(ax, i, j * len(model_order) + k, nrows, ncols)

    fig.tight_layout(rect=tight_layout_rect)
    fig.subplots_adjust(bottom=bottom, wspace=wspace)
    cbar_ax = fig.add_axes(colorbar_axes)
    cbar = fig.colorbar(cntr, cax=cbar_ax, orientation=colorbar_orientation)
    cbar.set_label(metric, fontsize=colorbar_label_size, fontweight=fweight)
    cbar.ax.tick_params(labelsize=colorbar_tick_size)

    figpath = FIGURE_OUTDIR / figname
    mov.save_figure(
        fig, figpath,
        mode="",
        metric="leadtime_acc",
        title=f"Leadtime ACC Skill: {field}",
        caption="ACC skill vs lead month",
        dpi=dpi,
    )
    print("Saved figure:", figpath)
    plt.show()


### Test significance of CESM-SMYLE vs E3SMLE, accounting for finite ensemble size
- Uses the common verification years for both systems.
- Set `E3SM_COMPARE_CASE_SELECTION` to one case, a list of cases, or `"all"`.
- Following the Yeager et al. method, resamples the larger ensemble down to the size of the smaller ensemble. Here CESM-SMYLE has 20 members and E3SMLE has 10, so CESM-SMYLE is repeatedly sampled at 10 members without replacement.
- Compares the resulting CESM-SMYLE ACC distribution with the full 10-member E3SMLE ACC.


In [ ]:
%%time
# CESM-SMYLE vs each E3SM hindcast ACC significance figure.
#
# Scientific purpose:
# 1. Restrict CESM-SMYLE and E3SM to common verification years.
# 2. Resample the larger CESM-SMYLE ensemble to the E3SM ensemble size.
# 3. Repeat the CESM-SMYLE subsampling N times without replacement.
# 4. Compute accpval = fraction of resampled CESM-SMYLE ACC > fixed E3SM ACC.

compare_cfg = WORKFLOW_SETTINGS["finite_ensemble_compare"]
compare_mode = compare_cfg["mode"]
compare_force_recompute = compare_cfg["force_recompute"]
compare_detrend = compare_cfg["detrend"]
compare_random_seed = compare_cfg["random_seed"]
cache_compare_anomalies = compare_cfg["cache_anomalies"]
persist_compare_inputs = compare_cfg["persist_inputs"]

# Select which configured E3SM hindcasts to compare with CESM-SMYLE.
# Use "all" for every configured case, a string for one case, or a list for many.
E3SM_COMPARE_CASE_SELECTION = compare_cfg["case_selection"]
if E3SM_COMPARE_CASE_SELECTION == "all":
    E3SM_COMPARE_CASES = list(E3SM_CASES)
elif isinstance(E3SM_COMPARE_CASE_SELECTION, str):
    E3SM_COMPARE_CASES = [E3SM_COMPARE_CASE_SELECTION]
else:
    E3SM_COMPARE_CASES = list(E3SM_COMPARE_CASE_SELECTION)

missing_compare_cases = [case_key for case_key in E3SM_COMPARE_CASES if case_key not in E3SM_CASES]
if missing_compare_cases:
    raise ValueError(f"E3SM_COMPARE_CASE_SELECTION includes cases not in E3SM_CASES: {missing_compare_cases}")
if not E3SM_COMPARE_CASES:
    raise ValueError("E3SM_COMPARE_CASE_SELECTION did not resolve to any E3SM cases.")

print(f"E3SM comparison cases: {E3SM_COMPARE_CASES}")

compare_modes = compare_cfg["modes"]
if compare_mode not in compare_modes:
    raise ValueError(f"compare_mode must be one of {list(compare_modes)}")

compare_mode_cfg = compare_modes[compare_mode]
compare_init_months = [m for m in compare_mode_cfg["init_months"] if m in init_months]
compare_iterations = compare_mode_cfg["iterations"]
compare_lead_start = compare_mode_cfg["lead_start"]
compare_lead_end = compare_mode_cfg["lead_end"]

if not compare_init_months:
    raise ValueError("No requested compare_init_months are available in init_months.")

# Dedicated chunks for the significance calculation. Do not reuse the
# earlier skill chunks, which combine all leads into each task.
compare_chunk_model = _copy_chunks("finite_ensemble_compare", "model_chunks")
compare_chunk_obs = _copy_chunks("finite_ensemble_compare", "obs_chunks")
compare_iteration_batch_size = compare_cfg["iteration_batch_size"]

obs_compare = obs_seas_rg.chunk(compare_chunk_obs)

def _compare_dir_e3sm(case_key, category):
    path = e3sm_leadtime_dir(E3SM_CASES[case_key], "comparison", category, field)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


def _compare_dir_smyle(category):
    path = smyle_leadtime_dir("comparison", category, field)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


def _compare_resampling_cache_dir_smyle(category):
    path = smyle_leadtime_dir("resampling_cache", category, field)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


def _safe_to_netcdf(ds, path):
    """Backward-compatible wrapper for the shared atomic NetCDF writer."""
    safe_to_netcdf(ds, path)


def _open_dataset_load(path):
    """
    Open a small cached result and load it into memory.
    Used for final skill / p-value outputs, not large anomaly inputs.
    """
    with xr.open_dataset(path) as ds:
        return ds.load()


def _cache_compare_anomaly(
    da,
    path,
    varname="anom",
    chunks=None,
    force=False,
):
    """
    Save a comparison anomaly DataArray once and reopen it with Dask chunks.

    This reduces the Dask graph size because the expensive significance
    calculation starts from NetCDF reads rather than from the full upstream
    lazy-processing graph.
    """
    if force or not os.path.exists(path):
        print(f"Writing cached anomaly: {path}")
        _safe_to_netcdf(da.to_dataset(name=varname), path)

    print(f"Opening cached anomaly with Dask chunks: {path}")
    return xr.open_dataset(path, chunks=chunks)[varname]


def _case_cache_tag(case_key):
    return E3SM_CASES[case_key].get("cache_tag", case_key.replace("E3SM-", ""))


def _compare_cache_paths(case_key, init_month, ens_size, years):
    base = f"{field}_clim_{climy0}_{climy1}"
    trend = "detrend" if compare_detrend else "nodetrend"
    year_token = f"{str(years[0])[:4]}-{str(years[-1])[:4]}_ny{len(years)}"
    mode = f"{compare_mode}_n{ens_size}_iter{compare_iterations}_seed{compare_random_seed}_{year_token}"
    case_tag = _case_cache_tag(case_key)

    smyle_file = os.path.join(
        _compare_dir_smyle("resampled_skill"),
        f"BSMYLE{init_month:02d}_{base}_{smyle_skill_cache_version}_resamp_to_{case_tag}_skill_{trend}_{mode}.nc",
    )

    e3sm_file = os.path.join(
        _compare_dir_e3sm(case_key, "fixed_skill"),
        f"{case_tag}{init_month:02d}_{base}_fixed_skill_{trend}.nc",
    )

    accp_file = os.path.join(
        _compare_dir_e3sm(case_key, "fraction_gt_smyle"),
        f"BSMYLE_gt_{case_tag}{init_month:02d}_{base}_{smyle_skill_cache_version}_fraction_{trend}_{mode}.nc",
    )

    return smyle_file, e3sm_file, accp_file


def _compare_anomaly_cache_paths(case_key, init_month, years):
    base = f"{field}_clim_{climy0}_{climy1}"
    trend = "detrend" if compare_detrend else "nodetrend"
    year_token = f"{str(years[0])[:4]}-{str(years[-1])[:4]}_ny{len(years)}"
    case_tag = _case_cache_tag(case_key)

    smyle_anom_file = os.path.join(
        _compare_dir_smyle("anomalies"),
        f"BSMYLE{init_month:02d}_{base}_{smyle_skill_cache_version}_compare_anom_{trend}_{year_token}.nc",
    )

    e3sm_anom_file = os.path.join(
        _compare_dir_e3sm(case_key, "anomalies"),
        f"{case_tag}{init_month:02d}_{base}_compare_anom_{trend}_{year_token}.nc",
    )

    return smyle_anom_file, e3sm_anom_file


def _compare_batch_cache_path(smyle_file, batch_start, batch_stop):
    basename = os.path.basename(smyle_file)
    stem, suffix = os.path.splitext(basename)
    return os.path.join(
        _compare_resampling_cache_dir_smyle("resampling"),
        f"{stem}_batch_{batch_start:03d}_{batch_stop:03d}{suffix}",
    )


def _acc_superiority_fraction(smyle_corr, e3sm_corr):
    """Fraction of valid SMYLE iterations exceeding E3SM ACC."""
    valid = smyle_corr.notnull() & e3sm_corr.notnull()
    return (smyle_corr > e3sm_corr).where(valid).mean("iteration", skipna=True)


rng = np.random.default_rng(compare_random_seed)

smyle_compare_skill_by_case_month = {}
e3sm_compare_skill_by_case_month = {}
accpval_by_case_month = {}
compare_years_by_case_month = {}
compare_ens_size_by_case_month = {}

for case_key in E3SM_COMPARE_CASES:
    smyle_compare_skill_by_case_month[case_key] = {}
    e3sm_compare_skill_by_case_month[case_key] = {}
    accpval_by_case_month[case_key] = {}
    compare_years_by_case_month[case_key] = {}
    compare_ens_size_by_case_month[case_key] = {}

    for compare_init_month in compare_init_months:
        e3sm_compare_member_count = anom_by_case_month[case_key][compare_init_month].sizes["M"]
        smyle_compare_member_count = smyle_anom_by_month[compare_init_month].sizes["M"]

        if smyle_compare_member_count < e3sm_compare_member_count:
            raise ValueError(
                "This comparison expects CESM-SMYLE to be the larger ensemble; "
                f"got SMYLE={smyle_compare_member_count}, {case_key}={e3sm_compare_member_count}."
            )
        compare_ens_size = e3sm_compare_member_count
        compare_ens_size_by_case_month[case_key][compare_init_month] = compare_ens_size

        compare_years = np.intersect1d(
            smyle_anom_by_month[compare_init_month].Y.values,
            anom_by_case_month[case_key][compare_init_month].Y.values,
        )
        compare_years_by_case_month[case_key][compare_init_month] = compare_years

        smyle_file, e3sm_file, accp_file = _compare_cache_paths(
            case_key, compare_init_month, compare_ens_size, compare_years
        )

        if (
            not compare_force_recompute
            and os.path.exists(smyle_file)
            and os.path.exists(e3sm_file)
            and os.path.exists(accp_file)
        ):
            print(f"Loading cached significance results for {case_key}, init_month={compare_init_month}")

            smyle_skill_tmp = _open_dataset_load(smyle_file)
            e3sm_skill_tmp = _open_dataset_load(e3sm_file)

            with xr.open_dataset(accp_file) as ds:
                accpval_tmp = ds["accpval"].load()

            # Older caches encoded an all-missing comparison as 0%. Repair
            # only those invalid cells from the already-cached skill arrays;
            # this does not repeat the expensive resampling calculation.
            valid_comparison = (
                smyle_skill_tmp.corr.notnull()
                & e3sm_skill_tmp.corr.notnull()
            ).any("iteration")
            invalid_cached = accpval_tmp.notnull() & ~valid_comparison
            accpval_tmp = accpval_tmp.where(valid_comparison)
            if bool(invalid_cached.any()):
                print(f"Repairing missing-value mask in {accp_file}")
                _safe_to_netcdf(accpval_tmp.to_dataset(name="accpval"), accp_file)

        else:
            print(f"Computing significance results for {case_key}, init_month={compare_init_month}")
            print(f"compare_mode       : {compare_mode}")
            print(f"compare_iterations : {compare_iterations}")
            print(f"compare_leads      : {compare_lead_start} to {compare_lead_end}")
            print(f"compare_detrend    : {compare_detrend}")
            print(f"cache anomalies    : {cache_compare_anomalies}")
            print(f"persist inputs     : {persist_compare_inputs}")

            smyle_compare_anom = (
                smyle_anom_by_month[compare_init_month]
                .sel(Y=compare_years)
                .chunk(compare_chunk_model)
            )

            e3sm_compare_anom = (
                anom_by_case_month[case_key][compare_init_month]
                .sel(Y=compare_years)
                .chunk(compare_chunk_model)
            )

            if cache_compare_anomalies:
                smyle_anom_file, e3sm_anom_file = _compare_anomaly_cache_paths(
                    case_key, compare_init_month, compare_years
                )

                smyle_compare_anom = _cache_compare_anomaly(
                    smyle_compare_anom,
                    smyle_anom_file,
                    varname="anom",
                    chunks=compare_chunk_model,
                    force=compare_force_recompute,
                )

                e3sm_compare_anom = _cache_compare_anomaly(
                    e3sm_compare_anom,
                    e3sm_anom_file,
                    varname="anom",
                    chunks=compare_chunk_model,
                    force=compare_force_recompute,
                )

            elif persist_compare_inputs:
                smyle_compare_anom = smyle_compare_anom.persist()
                e3sm_compare_anom = e3sm_compare_anom.persist()
                wait([smyle_compare_anom, e3sm_compare_anom])

            smyle_compare_time = smyle_time_by_month[compare_init_month].sel(
                Y=compare_years
            )

            e3sm_compare_time = time_by_case_month[case_key][compare_init_month].sel(
                Y=compare_years
            )

            if os.path.exists(e3sm_file) and not compare_force_recompute:
                print(f"Loading cached fixed {case_key} comparison skill...")
                e3sm_skill_tmp = _open_dataset_load(e3sm_file)
            else:
                print(f"Computing fixed {case_key} comparison skill...")
                e3sm_skill_tmp = stats.compute_skill_seasonal(
                    e3sm_compare_anom,
                    e3sm_compare_time,
                    obs_compare,
                    str(climy0),
                    str(climy1),
                    compare_lead_start,
                    compare_lead_end,
                    resamp=0,
                    detrend=compare_detrend,
                ).load()
                _safe_to_netcdf(e3sm_skill_tmp, e3sm_file)

            print("Computing CESM-SMYLE resampled skill distribution...")

            member_indices_all = np.stack([
                rng.choice(
                    smyle_compare_member_count,
                    size=compare_ens_size,
                    replace=False,
                )
                for _ in range(compare_iterations)
            ])

            smyle_skill_batches = []
            for batch_start in range(0, compare_iterations, compare_iteration_batch_size):
                batch_stop = min(batch_start + compare_iteration_batch_size, compare_iterations)
                batch_file = _compare_batch_cache_path(
                    smyle_file, batch_start, batch_stop
                )
                if os.path.exists(batch_file) and not compare_force_recompute:
                    print(f"  loading iterations {batch_start + 1}-{batch_stop}")
                    batch_skill = _open_dataset_load(batch_file)
                else:
                    print(f"  computing iterations {batch_start + 1}-{batch_stop}")
                    batch_skill = stats.compute_skill_seasonal_batch(
                        smyle_compare_anom,
                        smyle_compare_time,
                        obs_compare,
                        str(climy0),
                        str(climy1),
                        member_indices_all[batch_start:batch_stop],
                        compare_lead_start,
                        compare_lead_end,
                        detrend=compare_detrend,
                        metrics=("corr",),
                    ).load()
                    batch_skill = batch_skill.assign_coords(
                        iteration=np.arange(batch_start, batch_stop)
                    )
                    _safe_to_netcdf(batch_skill, batch_file)
                smyle_skill_batches.append(batch_skill)

            smyle_skill_tmp = xr.concat(smyle_skill_batches, dim="iteration")

            print("Computing ACC superiority fraction...")

            accpval_tmp = _acc_superiority_fraction(
                smyle_skill_tmp.corr, e3sm_skill_tmp.corr
            ).load()

            print(f"Writing significance cache for {case_key}, init_month={compare_init_month}")

            _safe_to_netcdf(smyle_skill_tmp, smyle_file)
            _safe_to_netcdf(accpval_tmp.to_dataset(name="accpval"), accp_file)

        smyle_compare_skill_by_case_month[case_key][compare_init_month] = smyle_skill_tmp
        e3sm_compare_skill_by_case_month[case_key][compare_init_month] = e3sm_skill_tmp
        accpval_by_case_month[case_key][compare_init_month] = accpval_tmp

        print("")
        print("case_key:", case_key)
        print("compare_init_month:", compare_init_month)
        print(
            "compare_years:",
            int(compare_years[0]),
            int(compare_years[-1]),
            "n=",
            compare_years.size,
        )
        print(f"{case_key} members:", e3sm_compare_member_count)
        print("CESM-SMYLE members:", smyle_compare_member_count)
        print("resampled ensemble size:", compare_ens_size)
        print(f"{case_key} fixed-ensemble skill:")
        print(e3sm_skill_tmp)
        print("CESM-SMYLE resampled skill distribution:")
        print(smyle_compare_skill_by_case_month[case_key][compare_init_month])


# Backward-compatible aliases for downstream plotting.
# Prefer November if available; otherwise use the last processed initialization month.
if 11 in compare_init_months:
    compare_init_month = 11
else:
    compare_init_month = compare_init_months[-1]

compare_case = E3SM_REFERENCE_CASE
compare_years = compare_years_by_case_month[compare_case][compare_init_month]
smyle_compare_skill_by_month = smyle_compare_skill_by_case_month[compare_case]
e3sm_compare_skill_by_month = e3sm_compare_skill_by_case_month[compare_case]
accpval_by_month = accpval_by_case_month[compare_case]
compare_ens_size_by_month = compare_ens_size_by_case_month[compare_case]
compare_years_by_month = compare_years_by_case_month[compare_case]

smyle_compare_skill = smyle_compare_skill_by_month[compare_init_month]
e3sm_compare_skill = e3sm_compare_skill_by_month[compare_init_month]
accpval = accpval_by_month[compare_init_month]

lon2d, lat2d = np.meshgrid(destgrid.lon, destgrid.lat)



In [ ]:
# -----------------------------------------------------------------------------
# Final ACC comparison figure setup
# -----------------------------------------------------------------------------
# Values come from WORKFLOW_SETTINGS["final_acc_figures"].
final_acc_cfg = WORKFLOW_SETTINGS["final_acc_figures"]
final_geom_cfg = final_acc_cfg["geometry"]
final_marker_cfg = final_acc_cfg["markers"]
final_acc_style_cfg = final_acc_cfg["acc"]
final_skilldiff_style_cfg = final_acc_cfg["skilldiff"]

compare_init_months_requested = list(final_acc_cfg["init_months"])
compare_plot_max_leads = final_acc_cfg["max_leads"]

# Panel geometry and spacing
compare_fig_col_width = final_geom_cfg["fig_col_width"]
compare_fig_row_height = final_geom_cfg["fig_row_height"]
compare_section_gap = final_geom_cfg["section_gap"]
compare_section_gap_fig_width = final_geom_cfg["section_gap_fig_width"]
compare_tight_layout_rect = list(final_geom_cfg["tight_layout_rect"])
compare_subplots_top = final_geom_cfg["subplots_top"]
compare_subplots_bottom = final_geom_cfg["subplots_bottom"]
compare_subplots_hspace = final_geom_cfg["subplots_hspace"]
compare_subplots_wspace = final_geom_cfg["subplots_wspace"]
compare_month_header_y = final_geom_cfg["month_header_y"]
compare_divider_y0 = final_geom_cfg["divider_y0"]
compare_divider_y1 = final_geom_cfg["divider_y1"]
compare_divider_color = final_geom_cfg["divider_color"]
compare_divider_linewidth = final_geom_cfg["divider_linewidth"]
compare_cbar_rect = list(final_geom_cfg["cbar_rect"])

# Shared map and significance-marker settings
compare_sig_level = final_marker_cfg["sig_level"]
compare_latlim = final_marker_cfg["latlim"]
compare_marker_color = final_marker_cfg["marker_color"]
compare_marker_stride = final_marker_cfg["marker_stride"]
compare_open_marker_size = final_marker_cfg["open_marker_size"]
compare_filled_marker_size = final_marker_cfg["filled_marker_size"]
compare_marker_linewidth = final_marker_cfg["marker_linewidth"]
compare_grid_linewidth = final_marker_cfg["grid_linewidth"]
compare_grid_color = final_marker_cfg["grid_color"]
compare_grid_alpha = final_marker_cfg["grid_alpha"]
compare_tick_length = final_marker_cfg["tick_length"]
compare_tick_width = final_marker_cfg["tick_width"]
compare_label_bbox = dict(final_marker_cfg["label_bbox"])

# ACC figure contents and styling
acc_include_cesm_smyle = final_acc_style_cfg["include_cesm_smyle"]
acc_cesm_label = final_acc_style_cfg["cesm_label"]
acc_ci = final_acc_style_cfg["ci"]
acc_cmin = final_acc_style_cfg["cmin"]
acc_cmax = final_acc_style_cfg["cmax"]
acc_cmap = final_acc_style_cfg["cmap"]
acc_cutoff = final_acc_style_cfg["cutoff"]
acc_fontz = final_acc_style_cfg["fontz"]
acc_panel_title_size = acc_fontz * 1.2
acc_lead_label_size = acc_fontz * 0.85
acc_lat_lon_label_size = acc_fontz * 0.8
acc_suptitle_size = acc_fontz * 1.25
acc_colorbar_label_size = acc_fontz
acc_colorbar_tick_size = acc_fontz * 0.85
acc_section_label_size = acc_fontz * 1.05

# E3SM minus matched CESM-SMYLE ACC figure styling
skilldiff_ci = final_skilldiff_style_cfg["ci"]
skilldiff_cmin = final_skilldiff_style_cfg["cmin"]
skilldiff_cmax = final_skilldiff_style_cfg["cmax"]
skilldiff_cmap = final_skilldiff_style_cfg["cmap"]
skilldiff_cutoff = final_skilldiff_style_cfg["cutoff"]
skilldiff_fontz = final_skilldiff_style_cfg["fontz"]
skilldiff_panel_title_size = skilldiff_fontz
skilldiff_lat_lon_label_size = skilldiff_fontz * 0.95
skilldiff_suptitle_size = skilldiff_fontz * 0.95
skilldiff_colorbar_label_size = skilldiff_fontz * 0.95
skilldiff_colorbar_tick_size = skilldiff_fontz * 0.90
skilldiff_lead_label_size = skilldiff_fontz * 0.90
skilldiff_section_label_size = skilldiff_fontz * 1.00

compare_fweight = final_acc_cfg["font_weight"]

# Figure titles and save metadata
acc_figure_title = final_acc_style_cfg["title_template"].format(plot_name=cfg["plot_name"], field=field)
acc_save_title = acc_figure_title
acc_save_caption = final_acc_style_cfg["caption"]
skilldiff_figure_title = final_skilldiff_style_cfg["title_template"].format(plot_name=cfg["plot_name"], field=field)
skilldiff_save_title = final_skilldiff_style_cfg["save_title_template"].format(plot_name=cfg["plot_name"], field=field)
skilldiff_save_caption = final_skilldiff_style_cfg["caption"]


In [ ]:
%%time
# Plot each selected E3SM hindcast ACC with ensemble-size-matched CESM-SMYLE
# significance markers. The figure columns are grouped by initialization month;
# selected E3SM cases are shown side by side within each month group.
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

ci = acc_ci
cmin = acc_cmin
cmax = acc_cmax
siglvl_compare = compare_sig_level
latlim = compare_latlim
symclr = compare_marker_color
marker_stride = compare_marker_stride
open_marker_size = compare_open_marker_size
filled_marker_size = compare_filled_marker_size
marker_linewidth = compare_marker_linewidth
compare_panel_title_size = acc_panel_title_size
compare_lead_label_size = acc_lead_label_size
compare_lat_lon_label_size = acc_lat_lon_label_size
compare_suptitle_size = acc_suptitle_size
compare_colorbar_label_size = acc_colorbar_label_size
compare_colorbar_tick_size = acc_colorbar_tick_size
compare_section_label_size = acc_section_label_size

compare_init_months_plot = [
    m for m in compare_init_months_requested
    if m in compare_init_months
    and m in smyle_compare_skill_by_case_month[E3SM_REFERENCE_CASE]
]
if not compare_init_months_plot:
    raise RuntimeError("No May/November initialization months are available for plotting.")

area = destgrid.area
valid_area = xr.where(abs(destgrid.lat) < latlim, area, 0)
valid_area_sum = valid_area.sum()
proj = ccrs.PlateCarree()

column_specs = []
for init_month in compare_init_months_plot:
    if acc_include_cesm_smyle and init_month in smyle_skill_by_month:
        column_specs.append((init_month, "CESM-SMYLE", None))

    for case_key in E3SM_COMPARE_CASES:
        if (
            init_month in smyle_compare_skill_by_case_month.get(case_key, {})
            and init_month in e3sm_compare_skill_by_case_month.get(case_key, {})
            and init_month in accpval_by_case_month.get(case_key, {})
        ):
            column_specs.append((init_month, "E3SM", case_key))

if not column_specs:
    raise RuntimeError("No CESM-SMYLE or E3SM case/month combinations are available for plotting.")


def _acc_skill_for_column(init_month, model_kind, case_key):
    if model_kind == "CESM-SMYLE":
        return smyle_skill_by_month[init_month]
    return e3sm_compare_skill_by_case_month[case_key][init_month]


plot_nlead = min(
    compare_plot_max_leads,
    *(
        _acc_skill_for_column(init_month, model_kind, case_key).sizes["L"]
        for init_month, model_kind, case_key in column_specs
    ),
)
nrows = plot_nlead
ncols = len(column_specs)
section_gap = compare_section_gap if len(compare_init_months_plot) > 1 else 0.0
fig = plt.figure(figsize=(compare_fig_col_width * ncols + compare_section_gap_fig_width * (section_gap > 0), compare_fig_row_height * nrows))
cntr = None


def _style_compare_axis(ax, row, col):
    ax.set_xticks([-120, -60, 0, 60, 120], crs=ccrs.PlateCarree())
    ax.set_yticks([-60, -30, 0, 30, 60], crs=ccrs.PlateCarree())
    if row == nrows - 1:
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
    else:
        ax.xaxis.set_major_formatter(plt.NullFormatter())

    if col == 0:
        ax.yaxis.set_major_formatter(LatitudeFormatter())
    else:
        ax.yaxis.set_major_formatter(plt.NullFormatter())
    ax.tick_params(
        labelsize=compare_lat_lon_label_size,
        length=compare_tick_length,
        width=compare_tick_width,
        top=False,
        right=False,
        labelbottom=(row == nrows - 1),
        labelleft=(col == 0),
    )
    ax.gridlines(
        crs=ccrs.PlateCarree(),
        linewidth=compare_grid_linewidth,
        color=compare_grid_color,
        alpha=compare_grid_alpha,
        linestyle="-",
        draw_labels=False,
    )


def _add_compare_markers(ax, base_skill, accp, lead_index):
    tmp = xr.where(~base_skill.corr.isel(L=lead_index).isnull(), accp.isel(L=lead_index), np.nan)

    # Open circles: at least 90% of matched-size CESM-SMYLE samples exceed E3SM.
    bsmyle_better = tmp > (1 - siglvl_compare)
    tmparea = xr.where(bsmyle_better, area, 0)
    tmparea = xr.where(abs(destgrid.lat) < latlim, tmparea, 0)
    nbetter = float(tmparea.sum() / valid_area_sum)
    display_mask = np.asarray(bsmyle_better)[::marker_stride, ::marker_stride]
    display_lon = lon2d[::marker_stride, ::marker_stride]
    display_lat = lat2d[::marker_stride, ::marker_stride]
    display_mask &= abs(display_lat) < latlim
    ax.scatter(
        display_lon[display_mask], display_lat[display_mask],
        facecolor="none", edgecolor=symclr,
        s=open_marker_size, linewidth=marker_linewidth, zorder=10,
    )

    # Filled dots: at most 10% of matched-size CESM-SMYLE samples exceed E3SM.
    e3sm_better = tmp < siglvl_compare
    tmparea = xr.where(e3sm_better, area, 0)
    tmparea = xr.where(abs(destgrid.lat) < latlim, tmparea, 0)
    nworse = float(tmparea.sum() / valid_area_sum)
    display_mask = np.asarray(e3sm_better)[::marker_stride, ::marker_stride]
    display_mask &= abs(display_lat) < latlim
    ax.scatter(
        display_lon[display_mask], display_lat[display_mask],
        facecolor=symclr, edgecolor=symclr,
        s=filled_marker_size, linewidth=0, zorder=10,
    )

    ax.text(
        0.98, 0.05,
        f"({nbetter * 100:3.1f}%/{nworse * 100:3.1f}%)",
        fontsize=compare_lead_label_size,
        fontweight=compare_fweight,
        bbox=compare_label_bbox,
        zorder=10,
        transform=ax.transAxes,
        ha="right", va="bottom",
    )


for i in range(plot_nlead):
    for col, (compare_init_month, model_kind, case_key) in enumerate(column_specs):
        if model_kind == "CESM-SMYLE":
            case_label = acc_cesm_label
            plot_skill = smyle_skill_by_month[compare_init_month]
            accp = None
        else:
            case_label = E3SM_CASES[case_key].get("display_name", case_key)
            plot_skill = e3sm_compare_skill_by_case_month[case_key][compare_init_month]
            accp = accpval_by_case_month[case_key][compare_init_month]

        init_label = init_month_names.get(compare_init_month, f"{compare_init_month:02d}")
        lead = int(plot_skill.L.values[i])
        display_lead = lead - 2
        leadstr = f"lead {display_lead}: {seasonal_label(compare_init_month, display_lead)}"
        title = f"{case_label}" if i == 0 else ""
        subplot = i * ncols + col + 1

        ax, cntr = maps.map_pcolor_global_subplot(
            fig,
            plot_skill.corr.isel(L=i),
            plot_skill.lon,
            plot_skill.lat,
            ci, cmin, cmax,
            title,
            nrows, ncols, subplot,
            proj, cmap=acc_cmap, cutoff=acc_cutoff,
            fontsize=compare_panel_title_size,
        )
        _style_compare_axis(ax, i, col)
        if col == 0 or col == len(column_specs) // 2:
            ax.text(
                0.02, 0.05, leadstr,
                fontsize=compare_lead_label_size,
                fontweight=compare_fweight,
                bbox=compare_label_bbox,
                zorder=10,
                transform=ax.transAxes,
                ha="left", va="bottom",
            )
        if accp is not None:
            _add_compare_markers(ax, plot_skill, accp, i)

fig.suptitle(
    acc_figure_title,
    fontsize=compare_suptitle_size,
    fontweight=compare_fweight,
    y=0.995,
)
layout_rect = compare_tight_layout_rect.copy()
layout_rect[2] = layout_rect[2] - section_gap
fig.tight_layout(rect=layout_rect)
fig.subplots_adjust(top=compare_subplots_top, bottom=compare_subplots_bottom, hspace=compare_subplots_hspace, wspace=compare_subplots_wspace)

if section_gap:
    second_month_cols = [i for i, (m, _, __) in enumerate(column_specs) if m == compare_init_months_plot[1]]
    second_month_axes = [fig.axes[row * ncols + col] for row in range(nrows) for col in second_month_cols]
    for ax in second_month_axes:
        pos = ax.get_position()
        ax.set_position([pos.x0 + section_gap, pos.y0, pos.width, pos.height])

# Add May/November headers above their case columns.
for init_month in compare_init_months_plot:
    cols = [i for i, (m, _, __) in enumerate(column_specs) if m == init_month]
    if not cols:
        continue
    left = min(fig.axes[i].get_position().x0 for i in cols)
    right = max(fig.axes[i].get_position().x1 for i in cols)
    init_label = init_month_names.get(init_month, f"{init_month:02d}")
    fig.text(
        (left + right) / 2,
        compare_month_header_y,
        f"{init_label} initialization",
        ha="center", va="bottom",
        fontsize=compare_section_label_size,
        fontweight=compare_fweight,
    )

if len(compare_init_months_plot) > 1:
    first_month_cols = [i for i, (m, _, __) in enumerate(column_specs) if m == compare_init_months_plot[0]]
    second_month_cols = [i for i, (m, _, __) in enumerate(column_specs) if m == compare_init_months_plot[1]]
    if first_month_cols and second_month_cols:
        divider_x = (
            max(fig.axes[i].get_position().x1 for i in first_month_cols)
            + min(fig.axes[i].get_position().x0 for i in second_month_cols)
        ) / 2
        fig.add_artist(plt.Line2D([divider_x, divider_x], [compare_divider_y0, compare_divider_y1], transform=fig.transFigure, color=compare_divider_color, linewidth=compare_divider_linewidth))

cbar_ax = fig.add_axes(compare_cbar_rect)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation="horizontal")
cbar.set_label("ACC", fontsize=compare_colorbar_label_size, fontweight=compare_fweight)
cbar.ax.tick_params(labelsize=compare_colorbar_tick_size)

compare_figname = figure_filename(field, "acc_skill_compare")
compare_figpath = FIGURE_OUTDIR / compare_figname
mov.save_figure(
    fig, compare_figpath,
    mode="",
    metric="leadtime_acc_compare",
    title=acc_save_title,
    caption=acc_save_caption,
    dpi=300,
)
print("Saved figure:", compare_figpath)
plt.show()


### E3SM ACC With Side-by-Side Case Comparison
- Plots one ACC figure for the selected E3SM cases.
- Set `E3SM_COMPARE_CASE_SELECTION` upstream to one case, multiple cases, or `"all"`.
- Columns are grouped by initialization month: May on the left, November on the right.
- E3SM hindcasts are shown side by side within each month group.


In [ ]:
%%time
# Plot fixed 10-member E3SM ACC minus mean matched-size CESM-SMYLE ACC.
# Columns are grouped by initialization month; selected E3SM cases are shown
# side by side within each month group.
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

ci = skilldiff_ci
cmin = skilldiff_cmin
cmax = skilldiff_cmax
siglvl_compare = compare_sig_level
latlim = compare_latlim
symclr = compare_marker_color
marker_stride = compare_marker_stride
open_marker_size = compare_open_marker_size
filled_marker_size = compare_filled_marker_size
marker_linewidth = compare_marker_linewidth
compare_panel_title_size = skilldiff_panel_title_size
compare_lat_lon_label_size = skilldiff_lat_lon_label_size
compare_suptitle_size = skilldiff_suptitle_size
compare_colorbar_label_size = skilldiff_colorbar_label_size
compare_colorbar_tick_size = skilldiff_colorbar_tick_size
compare_lead_label_size = skilldiff_lead_label_size
compare_section_label_size = skilldiff_section_label_size

compare_init_months_plot = [
    m for m in compare_init_months_requested
    if m in compare_init_months
    and m in smyle_compare_skill_by_case_month[E3SM_REFERENCE_CASE]
]
if not compare_init_months_plot:
    raise RuntimeError("No May/November initialization months are available for plotting.")

area = destgrid.area
valid_area = xr.where(abs(destgrid.lat) < latlim, area, 0)
valid_area_sum = valid_area.sum()
proj = ccrs.PlateCarree()

e3sm_compare_skilldiff_by_case_month = {
    case_key: {
        m: e3sm_compare_skill_by_case_month[case_key][m]
        - smyle_compare_skill_by_case_month[case_key][m].mean("iteration")
        for m in compare_init_months_plot
        if m in smyle_compare_skill_by_case_month.get(case_key, {})
        and m in e3sm_compare_skill_by_case_month.get(case_key, {})
    }
    for case_key in E3SM_COMPARE_CASES
}

column_specs = []
for init_month in compare_init_months_plot:
    for case_key in E3SM_COMPARE_CASES:
        if (
            init_month in smyle_compare_skill_by_case_month.get(case_key, {})
            and init_month in e3sm_compare_skill_by_case_month.get(case_key, {})
            and init_month in accpval_by_case_month.get(case_key, {})
            and init_month in e3sm_compare_skilldiff_by_case_month.get(case_key, {})
        ):
            column_specs.append((init_month, case_key))

if not column_specs:
    raise RuntimeError("No E3SM case/month combinations are available for plotting.")

plot_nlead = min(
    compare_plot_max_leads,
    *(e3sm_compare_skilldiff_by_case_month[case_key][init_month].sizes["L"] for init_month, case_key in column_specs),
)
nrows = plot_nlead
ncols = len(column_specs)
section_gap = compare_section_gap if len(compare_init_months_plot) > 1 else 0.0
fig = plt.figure(figsize=(compare_fig_col_width * ncols + compare_section_gap_fig_width * (section_gap > 0), compare_fig_row_height * nrows))
cntr = None


def _style_compare_axis(ax, row, col):
    ax.set_xticks([-120, -60, 0, 60, 120], crs=ccrs.PlateCarree())
    ax.set_yticks([-60, -30, 0, 30, 60], crs=ccrs.PlateCarree())
    if row == nrows - 1:
        ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))
    else:
        ax.xaxis.set_major_formatter(plt.NullFormatter())

    if col == 0:
        ax.yaxis.set_major_formatter(LatitudeFormatter())
    else:
        ax.yaxis.set_major_formatter(plt.NullFormatter())
    ax.tick_params(
        labelsize=compare_lat_lon_label_size,
        length=compare_tick_length,
        width=compare_tick_width,
        top=False,
        right=False,
        labelbottom=(row == nrows - 1),
        labelleft=(col == 0),
    )
    ax.gridlines(
        crs=ccrs.PlateCarree(),
        linewidth=compare_grid_linewidth,
        color=compare_grid_color,
        alpha=compare_grid_alpha,
        linestyle="-",
        draw_labels=False,
    )


def _add_compare_markers(ax, base_skill, accp, lead_index):
    tmp = xr.where(~base_skill.corr.isel(L=lead_index).isnull(), accp.isel(L=lead_index), np.nan)

    # Open circles: at least 90% of matched-size CESM-SMYLE samples exceed E3SM.
    bsmyle_better = tmp > (1 - siglvl_compare)
    tmparea = xr.where(bsmyle_better, area, 0)
    tmparea = xr.where(abs(destgrid.lat) < latlim, tmparea, 0)
    nbetter = float(tmparea.sum() / valid_area_sum)
    display_mask = np.asarray(bsmyle_better)[::marker_stride, ::marker_stride]
    display_lon = lon2d[::marker_stride, ::marker_stride]
    display_lat = lat2d[::marker_stride, ::marker_stride]
    display_mask &= abs(display_lat) < latlim
    ax.scatter(
        display_lon[display_mask], display_lat[display_mask],
        facecolor="none", edgecolor=symclr,
        s=open_marker_size, linewidth=marker_linewidth, zorder=10,
    )

    # Filled dots: at most 10% of matched-size CESM-SMYLE samples exceed E3SM.
    e3sm_better = tmp < siglvl_compare
    tmparea = xr.where(e3sm_better, area, 0)
    tmparea = xr.where(abs(destgrid.lat) < latlim, tmparea, 0)
    nworse = float(tmparea.sum() / valid_area_sum)
    display_mask = np.asarray(e3sm_better)[::marker_stride, ::marker_stride]
    display_mask &= abs(display_lat) < latlim
    ax.scatter(
        display_lon[display_mask], display_lat[display_mask],
        facecolor=symclr, edgecolor=symclr,
        s=filled_marker_size, linewidth=0, zorder=10,
    )

    ax.text(
        0.98, 0.05,
        f"({nbetter * 100:3.1f}%/{nworse * 100:3.1f}%)",
        fontsize=compare_lead_label_size,
        fontweight=compare_fweight,
        bbox=compare_label_bbox,
        zorder=10,
        transform=ax.transAxes,
        ha="right", va="bottom",
    )


for i in range(plot_nlead):
    for col, (compare_init_month, case_key) in enumerate(column_specs):
        case_label = E3SM_CASES[case_key].get("display_name", case_key)
        init_label = init_month_names.get(compare_init_month, f"{compare_init_month:02d}")
        smyle_skill = smyle_compare_skill_by_case_month[case_key][compare_init_month]
        skilldiff = e3sm_compare_skilldiff_by_case_month[case_key][compare_init_month]
        accp = accpval_by_case_month[case_key][compare_init_month]
        lead = int(smyle_skill.L.values[i])
        display_lead = lead - 2
        leadstr = f"lead {display_lead}: {seasonal_label(compare_init_month, display_lead)}"
        title = f"{case_label}" if i == 0 else ""
        subplot = i * ncols + col + 1

        ax, cntr = maps.map_pcolor_global_subplot(
            fig,
            skilldiff.corr.isel(L=i),
            smyle_skill.lon,
            smyle_skill.lat,
            ci, cmin, cmax,
            title,
            nrows, ncols, subplot,
            proj, cmap=skilldiff_cmap, cutoff=skilldiff_cutoff,
            fontsize=compare_panel_title_size,
        )
        _style_compare_axis(ax, i, col)
        if col == 0 or col == len(column_specs) // 2:
            ax.text(
                0.02, 0.05, leadstr,
                fontsize=compare_lead_label_size,
                fontweight=compare_fweight,
                bbox=compare_label_bbox,
                zorder=10,
                transform=ax.transAxes,
                ha="left", va="bottom",
            )
        _add_compare_markers(ax, skilldiff, accp, i)

fig.suptitle(
    skilldiff_figure_title,
    fontsize=compare_suptitle_size,
    fontweight=compare_fweight,
    y=0.995,
)
layout_rect = compare_tight_layout_rect.copy()
layout_rect[2] = layout_rect[2] - section_gap
fig.tight_layout(rect=layout_rect)
fig.subplots_adjust(top=compare_subplots_top, bottom=compare_subplots_bottom, hspace=compare_subplots_hspace, wspace=compare_subplots_wspace)

if section_gap:
    second_month_cols = [i for i, (m, _) in enumerate(column_specs) if m == compare_init_months_plot[1]]
    second_month_axes = [fig.axes[row * ncols + col] for row in range(nrows) for col in second_month_cols]
    for ax in second_month_axes:
        pos = ax.get_position()
        ax.set_position([pos.x0 + section_gap, pos.y0, pos.width, pos.height])

# Add May/November headers above their case columns.
for init_month in compare_init_months_plot:
    cols = [i for i, (m, _) in enumerate(column_specs) if m == init_month]
    if not cols:
        continue
    left = min(fig.axes[i].get_position().x0 for i in cols)
    right = max(fig.axes[i].get_position().x1 for i in cols)
    init_label = init_month_names.get(init_month, f"{init_month:02d}")
    fig.text(
        (left + right) / 2,
        compare_month_header_y,
        f"{init_label} initialization",
        ha="center", va="bottom",
        fontsize=compare_section_label_size,
        fontweight=compare_fweight,
    )

if len(compare_init_months_plot) > 1:
    first_month_cols = [i for i, (m, _) in enumerate(column_specs) if m == compare_init_months_plot[0]]
    second_month_cols = [i for i, (m, _) in enumerate(column_specs) if m == compare_init_months_plot[1]]
    if first_month_cols and second_month_cols:
        divider_x = (
            max(fig.axes[i].get_position().x1 for i in first_month_cols)
            + min(fig.axes[i].get_position().x0 for i in second_month_cols)
        ) / 2
        fig.add_artist(plt.Line2D([divider_x, divider_x], [compare_divider_y0, compare_divider_y1], transform=fig.transFigure, color=compare_divider_color, linewidth=compare_divider_linewidth))

cbar_ax = fig.add_axes(compare_cbar_rect)
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation="horizontal")
cbar.set_label(r"${\Delta}$ACC", fontsize=compare_colorbar_label_size, fontweight=compare_fweight)
cbar.ax.tick_params(labelsize=compare_colorbar_tick_size)

skilldiff_figname = figure_filename(field, "acc_skill_diff")
skilldiff_figpath = FIGURE_OUTDIR / skilldiff_figname
mov.save_figure(
    fig, skilldiff_figpath,
    mode="",
    metric="leadtime_acc_diff",
    title=skilldiff_save_title,
    caption=skilldiff_save_caption,
    dpi=300,
)
print("Saved figure:", skilldiff_figpath)
plt.show()

# Backward-compatible reference-case difference dictionary.
e3sm_compare_skilldiff_by_month = {
    m: e3sm_compare_skill_by_case_month[E3SM_REFERENCE_CASE][m]
    - smyle_compare_skill_by_case_month[E3SM_REFERENCE_CASE][m].mean("iteration")
    for m in compare_init_months_plot
}


### CESM-SMYLE Sample Length Comparison: 50-Year (1970–2019) vs 39-Year Overlap (1980–2018)
- Compares CESM-SMYLE ACC skill computed over its full 50-year record (1970–2019) on the left against the 39-year common overlap period (1980–2018) on the right.
- Demonstrates how sample length and decadal variability impact skill scores and gray shading cutoff boundaries.



In [ ]:
%%time
# Plot CESM-SMYLE full 50-year record (1970-2019) vs 39-year common overlap period (1980-2018)
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import mov_utils as mov

smyle_records = {
    "CESM-SMYLE (1970–2019, 50yr)": smyle_skill_by_month,
    "CESM-SMYLE (1980–2018, 39yr)": smyle_overlap_skill_by_month,
}

plot_leads = [1, 2, 3, 4, 5, 6, 7]
plot_months = [5, 11]

nrows = len(plot_leads)
ncols = len(plot_months) * len(smyle_records)

fig = plt.figure(figsize=(final_geom_cfg["fig_col_width"] * ncols, final_geom_cfg["fig_row_height"] * nrows))
fig.suptitle(
    f"CESM-SMYLE {cfg['plot_name']} Skill: 50-Year (1970–2019) vs 39-Year Overlap (1980–2018)",
    fontsize=final_acc_style_cfg["fontz"] * 1.25,
    fontweight="bold",
    y=0.995,
)

proj = ccrs.PlateCarree()
cntr = None

for i, lead in enumerate(plot_leads):
    for j, init_month in enumerate(plot_months):
        for k, (rec_label, rec_dict) in enumerate(smyle_records.items()):
            skill_ds = rec_dict[init_month]
            subplot = i * ncols + j * len(smyle_records) + k + 1
            title = f"{rec_label} ({init_month_names.get(init_month, f'{init_month:02d}')})" if i == 0 else ""

            ax, cntr = maps.map_pcolor_global_subplot(
                fig,
                skill_ds.corr.isel(L=i),
                skill_ds.lon,
                skill_ds.lat,
                acc_ci, acc_cmin, acc_cmax,
                title,
                nrows, ncols, subplot,
                proj, cmap=acc_cmap, cutoff=acc_cutoff,
                fontsize=final_acc_style_cfg["fontz"] * 1.1,
            )

fig.tight_layout(rect=[0.0, 0.05, 1.0, 0.96])
cbar_ax = fig.add_axes([0.3, 0.02, 0.4, 0.015])
cbar = fig.colorbar(cntr, cax=cbar_ax, orientation="horizontal")
cbar.set_label("ACC", fontsize=final_acc_style_cfg["fontz"], fontweight="bold")

smyle_comp_figname = figure_filename(field, "cesm_smyle_50yr_vs_39yr")
smyle_comp_figpath = FIGURE_OUTDIR / smyle_comp_figname
mov.save_figure(
    fig, smyle_comp_figpath,
    mode="",
    metric="leadtime_acc_smyle_record_length",
    title=f"CESM-SMYLE Record Length Comparison: {field}",
    caption="CESM-SMYLE 50-year vs 39-year ACC comparison",
    dpi=300,
)
print("Saved figure:", smyle_comp_figpath)
plt.show()

